# 01 - Initial Data Discovery (Interim Stage)

## 📌 Objective
Analyze the **Interim Level Data** to evaluate data uniqueness, redundancy, and baseline sparsity. This is the first deep look after raw JSONs are geocoded.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

%matplotlib inline
sns.set_theme(style="whitegrid")

s_path = Path("../data/interim/sales_geocoded.parquet")
df = pd.read_parquet(s_path) if s_path.exists() else None
print(f"Loaded {len(df) if df is not None else 0:,} Sales records.")

Loaded 19,273 Sales records.


## 🔎 Deep Analysis: Data Uniqueness
Real estate scrapers often capture the same listing multiple times if posted by different brokers. We use `property_hash` (derived from URL) as the primary key, but we also check for 'Semantic Duplicates' (same title, price, and area).

In [ ]:
if df is not None:
    total = len(df)
    unique_hashes = df['property_hash'].nunique()
    semantic_dupes = df.duplicated(subset=['title', 'price', 'area']).sum()
    
    print(f"Total Records: {total:,}")
    print(f"Unique Hashes: {unique_hashes:,} ({total - unique_hashes} drops)")
    print(f"Semantic Duplicates (Exact Title/Price/Area): {semantic_dupes:,}")
    
    # Strategic Decision: We deduplicate by hash first, then handle semantic overlaps via spatial clustering.

KeyError: 'property_hash'

## 🔎 Deep Analysis: Redundant Features
Identify columns that carry no signal for an ML model or are duplicated by numeric versions.

In [ ]:
redundant_cols = ['price_text', 'area_text', 'bhk_text', 'description_raw', 'listing_mode']
existing_redundant = [c for c in redundant_cols if c in df.columns]
print(f"Redundant Columns Detected: {existing_redundant}")
print("Reason: These are string representations of data we already have in numeric form (area, bedrooms, price).")

## 🔎 Deep Analysis: Outlier Containment
Visualizing the 'Long Tail' of Price and Area that justifies our clipping strategy.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(15, 5))
sns.boxplot(x=df['price'], ax=ax[0], color='salmon')
ax[0].set_title("Distribution of Raw Price (All Cities)")
ax[0].set_xscale('log')

sns.boxplot(x=df['area'], ax=ax[1], color='skyblue')
ax[1].set_title("Distribution of Raw Area (All Cities)")
ax[1].set_xscale('log')
plt.show()